# 🌍 Air Quality — EDA & Feature Engineering

**Objective:** Understand and prepare the raw training data for predicting mean daily air quality (`target`) across African monitoring stations, using satellite-derived atmospheric measurements and meteorological features.

---

## 📋 Notebook Structure
| Section | Description |
|---|---|
| 1. Imports | Libraries and global constants |
| 2. Data Loading | Load and inspect raw training data |
| 3. Exploratory Data Analysis | Missing values, station & temporal coverage |
| 4. Data Cleaning | Drop leaky/irrelevant columns, filter stations, handle NaNs |
| 5. Feature Engineering | Angle consolidation, AMF calculation, atmospheric indices |
| 6. Correlation Analysis | Feature-target correlation heatmaps before & after engineering |

---
## 1. 📦 Imports

We import all required libraries upfront for transparency:
- **Data manipulation:** `numpy`, `pandas`
- **Visualisation:** `matplotlib`, `seaborn`
- **Modelling:** `scikit-learn` — train/test split, linear regression, preprocessing, metrics
- `RSEED = 42` is set globally to ensure reproducibility.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

RSEED = 42

---
## 2. 📂 Data Loading

We load the training dataset from CSV. The dataset contains daily observations per monitoring station (`Place_ID`), covering **~30,000 rows** and **82 columns** including:
- **Target:** `target` — mean daily air quality (PM2.5 proxy)
- **Meteorological features:** temperature, humidity, wind components, precipitable water
- **Satellite features (L3):** NO₂, O₃, CO, HCHO, SO₂, aerosol indices, cloud properties, CH₄

The `Place_ID X Date` column acts as a composite row identifier.

In [ ]:
# Import diabetes data
df = pd.read_csv('./../Data/Train.csv')
df.head()

### 2.1 Dataset Overview
Inspect column types, dtypes, and non-null counts. Many satellite columns have significant missing data — this will be addressed in the cleaning step.

In [ ]:
df.info()

### 2.2 Missing Value Count
Check how many values are missing per column. Notice that CH₄ columns have very high missingness (~81%), and several satellite bands (SO₂, HCHO) also have substantial gaps.

In [ ]:
df.isnull().sum()

---
## 3. 🔍 Exploratory Data Analysis

Before modelling, we explore the structure of the dataset. Key questions:
- How many unique monitoring stations are there?
- What is the time coverage of the dataset?
- Are there stations with very few observations (potentially unreliable)?

### 3.1 Initial Data Cleaning: Drop Irrelevant Columns

We remove the following columns before further analysis:
- `Place_ID X Date` — composite ID, not a feature
- `target_min`, `target_max`, `target_variance`, `target_count` — derived from the target; would cause **data leakage**
- All `L3_CH4_*` columns — >81% missing data, insufficient for reliable imputation

In [ ]:
cols_to_drop = [
    "Place_ID X Date",
    "target_min",
    "target_max",
    "target_variance",
    "target_count",
    "L3_CH4_CH4_column_volume_mixing_ratio_dry_air",
    "L3_CH4_aerosol_height",
    "L3_CH4_aerosol_optical_depth",
    "L3_CH4_sensor_azimuth_angle",
    "L3_CH4_sensor_zenith_angle",
    "L3_CH4_solar_azimuth_angle",
    "L3_CH4_solar_zenith_angle"
]


df = df.drop(columns=cols_to_drop)
df.head()

### 3.2 Station Coverage
Check the number of observations per monitoring station (`Place_ID`). Stations with very few entries may need to be filtered out to prevent noise in time-series interpolation.

In [ ]:
df['Place_ID'].value_counts()

### 3.3 Temporal Coverage
Inspect the date range and observation count per day. The dataset spans ~94 days (Jan–Apr 2020), with roughly 320–335 stations reporting each day.

In [ ]:
df['Date'].value_counts().sort_index()

---
## 4. 🧹 Data Cleaning

Three-step cleaning pipeline:
1. **Replace zeros with NaN** — Sensor zeros are likely missing values in this atmospheric context
2. **Filter by station count** — Keep only stations with ≥ 40 observations (drops very sparse stations)
3. **Sort by station and date** — Required for coherent time-series interpolation

> **Note:** `groupby().filter()` is used to apply the station-level threshold efficiently.

In [ ]:
# 1. Filter: Keep only Place_IDs that have at least 40 entries
# We group by the column (axis=0 is default), not the headers (axis=1)
df = df.replace(0, np.nan)
df = df.groupby('Place_ID', as_index=False).filter(lambda x: len(x) >= 40)

# 2. Sort: Ensure time-series order for interpolation to make sense
# If 'Date' isn't a datetime object yet, you might want: df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Place_ID', 'Date'])

# 3. Interpolate: Fill missing values within each group
# We use group_keys=False to prevent the Place_ID from becoming a redundant index
df = df.groupby('Place_ID', as_index=False).apply(lambda x: x.interpolate(limit_direction='both'))

# Check the results
df['Place_ID'].value_counts()

### 4.1 Remaining Missing Values After Cleaning
Check null counts again after the filtering step.

In [ ]:
df.isnull().sum()

### 4.2 Cleaned Dataset Schema
The dataset now has ~30,500 rows across 340 stations with 70 features remaining.

In [ ]:
df.info()

---
## 5. ⚙️ Feature Engineering

The raw dataset contains many highly redundant angle columns (solar/sensor azimuth & zenith repeated across every satellite band). We apply several transformations to reduce dimensionality and create physically meaningful features.

### 5.1 Correlation with Target (Before Engineering)
We first plot the feature-target correlations to identify which satellite bands are most informative *before* any engineering. This also guides which features to consolidate.

In [ ]:
def plot_target_correlation(data, target_col, title="Feature Correlation with Target"):
    """Plots a sorted heatmap showing the correlation of all features with the target."""
    correlations = data.corr(numeric_only=True)[[target_col]]
    correlations = correlations.sort_values(by=target_col, ascending=False)
    correlations = correlations.drop(target_col)

    plt.figure(figsize=(15, 18))
    sns.heatmap(correlations, annot=True, cmap='viridis',
                vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
    plt.title(f"{title}: {target_col}")
    plt.show()

plot_target_correlation(df, "target")

### 5.2 Angle Consolidation — Relative Azimuth & Zenith

Each satellite band independently records solar/sensor azimuth and zenith angles. Since the sun position is the same for all instruments at a given time/location, these columns are **highly collinear**.

**Strategy:** Compute the mean across all bands and then derive:
- `relative_azimuth` = |mean solar azimuth − mean sensor azimuth|
- `relative_zenith` = |mean solar zenith − mean sensor zenith|

All original per-band angle columns are then dropped.

In [ ]:
def relative_mean_angles(df: pd.DataFrame):
    angle_map = {
        'solar_azimuth': [col for col in df.columns if 'solar_azimuth_angle' in col],
        'sensor_azimuth': [col for col in df.columns if 'sensor_azimuth_angle' in col],
        'solar_zenith': [col for col in df.columns if 'solar_zenith_angle' in col],
        'sensor_zenith': [col for col in df.columns if 'sensor_zenith_angle' in col]
    }

    mean_var = pd.DataFrame()
    for name, cols in angle_map.items():
        mean_var[f'mean_{name}'] = df[cols].mean(axis=1)

    df['relative_azimuth'] = np.abs(mean_var['mean_solar_azimuth'] - mean_var['mean_sensor_azimuth'])


    df['relative_zenith'] = np.abs(mean_var['mean_solar_zenith'] - mean_var['mean_sensor_zenith'])

    # Calculate Air Mass Factor Proxy (1 / cos(solar_zenith))
    # This is physically significant for light path length through the atmosphere
    #df['solar_zenith_rad'] = np.radians(df['mean_solar_zenith'])
    #df['air_mass_factor_proxy'] = 1 / np.cos(df['solar_zenith_rad'])

    # 5. Clean up: Remove all the original redundant angle columns
    all_original_angles = [col for list_of_cols in angle_map.values() for col in list_of_cols]
    df.drop(columns=all_original_angles, inplace=True)
    #df.drop(columns=all_original_angles + ['solar_zenith_rad'], inplace=True)

    print(f"Reduced features. New columns added: {list(df.columns[-6:])}")
    return df

df = relative_mean_angles(df)

Dataset schema after angle consolidation — 44 columns remain (down from 70).

In [ ]:
df.info()

### 5.3 Air Mass Factors (AMF) & Atmospheric Indices

We engineer physically meaningful ratio features:

| Feature | Formula | Physical Meaning |
|---|---|---|
| `AMF_NO2` | slant_NO2 / vertical_NO2 | Light path length through atmosphere for NO₂ |
| `AMF_SO2_calc` | slant_SO2 / vertical_SO2 | Same for SO₂ |
| `AMF_HCHO_calc` | slant_HCHO / tropospheric_HCHO | Air mass factor proxy for formaldehyde |
| `NO2_Tropo_Ratio` | tropospheric_NO2 / total_NO2 | Fraction of NO₂ in troposphere (pollution signal) |
| `Cloud_Thickness_Pressure` | cloud_base_pressure − cloud_top_pressure | Pressure depth of cloud layer |

We also reduce collinearity in repeated measurements:
- **`mean_cloud_fraction`** — average of all per-band cloud fraction columns (then drop originals)
- **`mean_sensor_altitude`** — average of all sensor altitude columns (then drop originals)

In [ ]:
def calculate_air_mass_factors(df):
    """
    Calculates the ratio between slant and vertical columns.
    Matches exact column names from your df.info().
    """
    # NO2 AMF
    df['AMF_NO2'] = df['L3_NO2_NO2_slant_column_number_density'] / df['L3_NO2_NO2_column_number_density']
    
    # SO2 AMF (Using the slant and column density)
    df['AMF_SO2_calc'] = df['L3_SO2_SO2_slant_column_number_density'] / df['L3_SO2_SO2_column_number_density']
    
    # HCHO AMF (Using slant and tropospheric column density)
    df['AMF_HCHO_calc'] = df['L3_HCHO_HCHO_slant_column_number_density'] / df['L3_HCHO_tropospheric_HCHO_column_number_density']
    
    return df

def calculate_atmospheric_indices(df):
    """
    Calculates environmental ratios and cloud metrics.
    """
    # NO2 Tropo Ratio: How much of the total NO2 is in the troposphere?
    df['NO2_Tropo_Ratio'] = df['L3_NO2_tropospheric_NO2_column_number_density'] / df['L3_NO2_NO2_column_number_density']
    
    # Cloud pressure thickness (Base - Top)
    df['Cloud_Thickness_Pressure'] = df['L3_CLOUD_cloud_base_pressure'] - df['L3_CLOUD_cloud_top_pressure']
    
    return df

def cloud_fraction_reduction(df: pd.DataFrame):
    """ Preventing feature co linearity by taking the mean of the cloud fraction

    Args:
        df (pd.DataFrame): original dataframe

    Returns:
        _type_: output dataframe with the old columns removed and the mean column added
    """
    cloud_frac_cols = [c for c in df.columns if 'cloud_fraction' in c]
    df['mean_cloud_fraction'] = df[cloud_frac_cols].mean(axis=1)
    df = df.drop(columns=cloud_frac_cols, axis = 1)

    return df
def sensor_altitude_reduction(df: pd.DataFrame):
    """ Preventing feature co linearity by taking the mean of the sensor altitudes

    Args:
        df (pd.DataFrame): original dataframe

    Returns:
        _type_: output dataframe with the old columns removed and the mean column added
    """
    sensor_altitude_cols = [c for c in df.columns if 'sensor_altitude' in c]
    df['mean_sensor_altitude'] = df[sensor_altitude_cols].mean(axis=1)
    df = df.drop(columns=sensor_altitude_cols, axis = 1)

    return df

df = cloud_fraction_reduction(df)
df = sensor_altitude_reduction(df)
# --- How to use them ---
df = calculate_air_mass_factors(df)
df = calculate_atmospheric_indices(df)

Dataset schema after feature engineering — 43 columns (engineered features added, collinear ones removed).

In [ ]:
df.info()

### 5.4 Final Column Drop
Remove remaining non-numeric identifier columns (`Date`, `Place_ID`) and structurally redundant cloud geometry columns (`L3_CLOUD_cloud_base_height`, `L3_CLOUD_cloud_base_pressure`) before modelling.

In [ ]:
cols_to_drop_2 = [
    "Date",
    "Place_ID",
    "L3_CLOUD_cloud_base_height",
    "L3_CLOUD_cloud_base_pressure"
]

df = df.drop(columns=cols_to_drop_2)
df

---
## 6. 📊 Correlation Analysis (After Feature Engineering)

Re-plot the feature-target correlation heatmap on the final engineered feature set. Compare with the pre-engineering heatmap to assess whether the new features have improved signal quality.

> Features with high positive/negative correlation are strong candidates for the model. Features near zero correlation may contribute noise.

In [ ]:
def plot_target_correlation(data, target_col, title="Feature Correlation with Target"):
    """Plots a sorted heatmap showing the correlation of all features with the target."""
    correlations = data.corr(numeric_only=True)[[target_col]]
    correlations = correlations.sort_values(by=target_col, ascending=False)
    correlations = correlations.drop(target_col)

    plt.figure(figsize=(15, 18))
    sns.heatmap(correlations, annot=True, cmap='viridis',
                vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
    plt.title(f"{title}: {target_col}")
    plt.show()

plot_target_correlation(df, "target")

---
## 7. 📝 EDA Summary

### What We Found
- The dataset covers **~30,500 daily observations** across **340 monitoring stations** over a 94-day window (Jan–Apr 2020)
- Several satellite bands (CH₄, SO₂, HCHO) had **high missingness** — CH₄ was dropped entirely; others were retained after filtering
- Replacing sensor zeros with `NaN` and interpolating within station groups produced a clean, complete feature matrix

### What We Engineered
| Feature | Rationale |
|---|---|
| `relative_azimuth`, `relative_zenith` | Collapsed ~32 redundant per-band angle columns into 2 physically meaningful ones |
| `AMF_NO2`, `AMF_SO2_calc`, `AMF_HCHO_calc` | Air mass factor ratios — encode light path length through atmosphere |
| `NO2_Tropo_Ratio` | Fraction of NO₂ in troposphere — a direct pollution signal |
| `Cloud_Thickness_Pressure` | Pressure depth of cloud layer — affects satellite retrieval quality |
| `mean_cloud_fraction`, `mean_sensor_altitude` | Averaged repeated measurements to reduce multicollinearity |

### Final Dataset
- **Rows:** ~30,500 &nbsp;|&nbsp; **Features:** 39 (down from 82 raw columns)
- All features are numeric and ready for modelling
- The correlation heatmap confirms that NO₂, CO, and meteorological variables carry the strongest signal relative to the target